In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from typing import Tuple, List, Optional


# =========================
# 1. Reward Model
# =========================
class RewardModel(nn.Module):
    """
    Reward model built on top of a pretrained transformer backbone.
    The backbone encodes the sequence; a linear head outputs a scalar reward.
    """
    def __init__(self, model_name: str = "bert-base-uncased", use_margin_loss: bool = False):
        super().__init__()
        # Load a pretrained backbone model
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        # Reward head: hidden_size -> 1 scalar
        self.reward_head = nn.Linear(hidden_size, 1)
        self.use_margin_loss = use_margin_loss

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the reward model.

        Expects encoder-style inputs, e.g.:
            input_ids, attention_mask, token_type_ids, ...

        We just forward all of them to the backbone.
        """
        outputs = self.backbone(input_ids, attention_mask=attention_mask)  # backbone handles token_type_ids etc.
        last_hidden_state = outputs.last_hidden_state  # [batch, seq_len, hidden_size]

        # Use the last token embedding (similar to EOS)
        eos_embedding = last_hidden_state[:, -1, :]    # [batch, hidden_size]

        # Reward scalar per sequence
        reward = self.reward_head(eos_embedding).squeeze(-1)  # [batch]
        return reward


In [4]:
model = RewardModel("bert-base-uncased")
batch_size, seq_len = 3, 7
input_ids = torch.randint(0, 1000, (batch_size, seq_len))
attention_mask = torch.ones(batch_size, seq_len, dtype=torch.long)

rewards = model(input_ids, attention_mask)
print("rewards:", rewards)
print("shape:", rewards.shape)  # should be [3]

rewards: tensor([0.4655, 0.3867, 0.4259], grad_fn=<SqueezeBackward1>)
shape: torch.Size([3])


In [5]:
# =========================
# 3. Pairwise Loss
# =========================
def pairwise_loss(
    chosen_rewards: torch.Tensor,
    rejected_rewards: torch.Tensor,
    margins: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """
    Pairwise preference loss used in reward modeling.
    
    Standard form:
        L = -log( sigmoid( r_chosen - r_rejected ) )

    If a margin tensor is provided, we use a margin-based variant:
        L = -log( sigmoid( (r_chosen - r_rejected) - margin ) )

    Args:
        chosen_rewards:   Tensor of shape [batch]
        rejected_rewards: Tensor of shape [batch]
        margins:          Optional tensor of shape [batch] for margin loss

    Returns:
        Scalar loss (mean over batch)
    """
    diff = chosen_rewards - rejected_rewards
    
    # Optional margin loss
    if margins is not None:
        diff = diff - margins

    return -torch.log(torch.sigmoid(diff)).mean()


In [6]:
# =========================
# 2. Preference Dataset
# =========================
class PreferenceDataset(Dataset):
    """
    Loads paired preference data in the format:
        (prompt, chosen_response, rejected_response)
    """
    def __init__(self, data: List[Tuple[str, str, str]], tokenizer: AutoTokenizer, max_length: int = 128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx) -> Tuple[dict, dict]:
        prompt, chosen, rejected = self.data[idx]
        
        # Encode the 'chosen' sequence:
        # [CLS] prompt [SEP] chosen_response [SEP]
        chosen_enc = self.tokenizer(
            prompt, 
            chosen,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Encode the 'rejected' sequence in the same way
        rejected_enc = self.tokenizer(
            prompt, 
            rejected,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return chosen_enc, rejected_enc


In [1]:
from transformers import AutoModel, AutoTokenizer

# =========================
# 4. Hyperparameters & Setup
# =========================
model_name = "Qwen/Qwen3-0.6B"   # or the exact HF ID you used originally

backbone = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,      # many Qwen models need this
    cache_dir="../02-DeepSeeK/hf_models",
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    cache_dir="../02-DeepSeeK/hf_models",
)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [11]:
# Training hyperparameters
# model_name = "Qwen/Qwen3-0.6B"           # <--- use HF ID, not snapshot hash
# cache_dir = "../02-DeepSeeK/hf_models"   # optional, for local caching
model_name = "Qwen/Qwen3-0.6B" 
cache_dir = "../02-DeepSeeK/hf_models" 

batch_size = 8
grad_accum_steps = 2      # Gradient accumulation (reduces memory usage)
use_amp = True            # Enable mixed-precision training if available
use_margin_loss = True    # Whether to apply margin-based pairwise loss

# Device selection
# or: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Device selection (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

# Initialize model and optimizer
model = RewardModel(model_name)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

Using device: cpu


In [12]:
# Example preference data (replace with your real dataset)
train_data = [
    ("Explain reinforcement learning.",
     "Reinforcement learning is a method where an agent learns through reward signals.",
     "Reinforcement learning is just an algorithm."),

    ("What is Newton's third law?",
     "Action and reaction forces are equal in magnitude and opposite in direction.",
     "Newton's laws are mainly about gravity."),
]

# Dataset & DataLoader
train_dataset = PreferenceDataset(train_data, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [14]:
# =========================
# 5. Training Loop
# =========================
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
model.train()

for epoch in range(1):
    total_loss = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(train_dataloader):
        chosen_batch, rejected_batch = batch
        chosen_inputs = {k:v.squeeze(1) for k,v in chosen_batch.items()}
        rejected_inputs = {k:v.squeeze(1) for k,v in rejected_batch.items()}
        
        with torch.cuda.amp.autocast(enabled=use_amp):
            r_chosen = model(**chosen_inputs)
            r_reject = model(**rejected_inputs)
            margins = torch.rand(len(r_chosen)).to(device) 
            loss = pairwise_loss(r_chosen, r_reject, margins)
        scaler.scale(loss).backward()
        total_loss+=loss.item()
        
        if (step + 1) % grad_accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        print(f"Epoch{epoch+1}, Loss: {total_loss/len(train_dataloader):.4f}")

/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_25243/2864216509.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/Users/linghuang/miniconda3/envs/llm/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_25243/2864216509.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/Users/linghuang/miniconda3/envs/llm/lib/python3.11/site-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Epoch1, Loss: 0.5396


In [15]:
# =========================
# 6. Save the trained reward model
# =========================
torch.save(model.state_dict(), "reward_model.pt")
print("Saved reward model to reward_model.pt")

Saved reward model to reward_model.pt
